# 🏛️ Pipeline completo M1 → M2 → M3 → M4 con evaluación con etiquetas (Colab)

Corre **todo el pipeline sobre una escena etiquetada** (por ejemplo la carpeta `listo.zip`: un `.las` por pieza, el nombre da la clase) y mide **mIoU** de cada módulo.

| Celda | Qué hace | Tarda (estimado) |
|---|---|---|
| 1 | GPU, Drive, carpetas y **parámetros** | segundos |
| 2 | Clona el repo (rama `main`) | segundos |
| 3-3b | Entorno de Sonata: **se restaura de Drive**; si la GPU cambió, recompila solo | 1-2 min (25 min la 1ª vez en una GPU nueva) |
| 4 | Une las piezas y guarda las etiquetas | ~1 min |
| 5 | **M1** preprocesa y transfiere las etiquetas | ~1 min |
| 6-7 | **M2** HyperNCD (entorno aparte, se restaura de Drive) | 1-3 min |
| 8 | **M3**: Sonata **original** y **adaptado** (u otros pesos) | unos minutos por variante |
| 9 | **M4** con cada M3 | ~1 min por variante y K |
| 10 | **Evaluación con etiquetas** y todo guardado en Drive | segundos |
| 11 | (opcional) **Control**: Sonata original en el mismo entorno que el adaptado | minutos |

Los tiempos son estimados, no medidos en Colab.

**Antes de empezar:** sube `listo.zip` a tu Drive en `PFC1_HyperNCD_Sonata/m1/entrada/etiquetado/listo.zip` (o la celda 4 te deja subirlo).
**Si se cae la sesión:** vuelve a correr desde la celda 1; lo ya calculado se toma de Drive y no se repite (pon `REHACER = True` para forzarlo).
**Cambiar de modelo:** edita `CKPT_M2` (otro checkpoint de HyperNCD) o `M3_VARIANTES` (otros pesos de Sonata) en la celda 1.

In [ ]:
# @title 1. GPU, Drive, carpetas y PARÁMETROS
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, torch
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
M1_ENT, M1_SAL = f"{DRIVE}/m1/entrada", f"{DRIVE}/m1/salida"
M2_ENT, M2_SAL = f"{DRIVE}/m2/entrada", f"{DRIVE}/m2/salida"
M3_ENTRENA, M3_PESOS, M3_SAL = f"{DRIVE}/m3/entrenamiento", f"{DRIVE}/m3/pesos", f"{DRIVE}/m3/salida"
M4_SAL   = f"{DRIVE}/m4/salida"
ENTORNOS = f"{DRIVE}/entornos"
for _d in (M1_ENT, M1_SAL, M2_ENT, M2_SAL, M3_ENTRENA, M3_PESOS, M3_SAL, M4_SAL, ENTORNOS):
    os.makedirs(_d, exist_ok=True)

def _mover(origen, destino):
    """Mueve origen -> destino solo si origen existe. Nunca pisa nada: si destino existe, no lo toca."""
    if not os.path.exists(origen):
        return
    if os.path.exists(destino):
        if os.path.isdir(origen) and os.path.isdir(destino):   # fusiona carpetas
            for x in os.listdir(origen):
                _mover(os.path.join(origen, x), os.path.join(destino, x))
            try:
                os.rmdir(origen)
            except OSError:
                pass
        else:
            print("  ya existe, no se toca:", destino.replace(DRIVE + "/", ""))
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.move(origen, destino)
    print("  movido:", origen.replace(DRIVE + "/", ""), "->", destino.replace(DRIVE + "/", ""))

# M1: la salida (los .npy preprocesados, empaquetados)
_mover(f"{DRIVE}/data_m1.tar.gz",    f"{M1_SAL}/data_m1.tar.gz")
_mover(f"{DRIVE}/m1/data_m1.tar.gz", f"{M1_SAL}/data_m1.tar.gz")
# M2: checkpoints de entrada y resultados
_mover(f"{DRIVE}/datos_m2_descomprimidos",         f"{M2_ENT}/datos_m2_descomprimidos")
_mover(f"{DRIVE}/resultados_m2",                   f"{M2_SAL}/resultados_m2")
_mover(f"{DRIVE}/resultados_modulo2_hyperncd.zip", f"{M2_SAL}/resultados_modulo2_hyperncd.zip")
# Entornos compilados
_mover(f"{DRIVE}/hyperncd_env_cache.tar.gz", f"{ENTORNOS}/hyperncd_env_cache.tar.gz")
for _p in glob.glob(f"{DRIVE}/pointcept_env_*.tar.gz"):
    _mover(_p, f"{ENTORNOS}/{os.path.basename(_p)}")
# M3: si ya existía la carpeta anterior de la adaptación
_mover(f"{DRIVE}/sonata_adaptado/entrenamiento",       M3_ENTRENA)
_mover(f"{DRIVE}/sonata_adaptado/sonata_catedral.pth", f"{M3_PESOS}/sonata_catedral.pth")
_mover(f"{DRIVE}/sonata_adaptado/features",            M3_SAL)
print("Estructura lista en", DRIVE)


# ===================== PARÁMETROS (edita aquí) =====================
ESCENA = "capilla_etiquetada"                                  # nombre de la escena (carpetas y archivos)
ZIP_ETIQUETADO = f"{M1_ENT}/etiquetado/listo.zip"              # zip con un .las por pieza (clase = nombre del archivo)
CKPT_M2 = "AUTO"                                               # "AUTO" = HyperNCD split0_run4 época 8; o la ruta a otro .ckpt
M3_VARIANTES = {                                               # nombre -> ruta de pesos (None = Sonata original de Meta)
    "original": None,
    "adaptado": f"{M3_PESOS}/sonata_catedral.pth",
    # "otro_modelo": "/content/drive/MyDrive/PFC1_HyperNCD_Sonata/m3/pesos/mis_pesos.pth",
}
K_LISTA = [6, 8, 12]                                           # grupos de M4 (6 = número de clases de la escena)
REHACER = False                                                # True = ignora lo guardado en Drive y recalcula
BRANCH = "main"
# ====================================================================
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
OUT_DRIVE = f"{DRIVE}/m4/salida/{ESCENA}"; os.makedirs(OUT_DRIVE, exist_ok=True)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
# @title 2. Clonar / actualizar tu repo (rama indicada en BRANCH)
import os
from getpass import getpass

REPO_OWNER, REPO_NAME = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata"
REPO_DIR = f"/content/{REPO_NAME}"

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        print("✅ Token leído desde Colab Secrets (GITHUB_TOKEN).")
except Exception:
    pass
if not token:
    token = getpass("Ingresa tu GitHub Personal Access Token (oculto): ").strip()

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
%cd {REPO_DIR}
!git log -1 --oneline
for f in ("experimentacion/evaluacion_etiquetada/preparar_etiquetas.py", "experimentacion/evaluacion_etiquetada/evaluar_m4.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"Falta {f}: haz commit y push de esos archivos a la rama {BRANCH}."

## Entorno de Sonata / M1 / M3 (se restaura de Drive)
La celda 3 busca `entornos/pointcept_env_<python>_<torch>_<cuda>.tar.gz` en tu Drive. Si existe, lo descomprime (1-2 min) en vez de reinstalar. Si no, lo construye (15-25 min) y lo guarda. La 3b comprueba que las extensiones compiladas funcionen **en la GPU de esta sesión** y, si no (por ejemplo cambiaste de T4 a L4), las recompila y actualiza el archivo de Drive.

In [ ]:
# @title 3. Entorno de Pointcept (con caché en Drive: la 1ª vez compila, las siguientes restauran)
import os, sys, subprocess, torch

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló (código {r.returncode}): {cmd}")

PYV  = f"py{sys.version_info.major}{sys.version_info.minor}"
TV   = torch.__version__.split("+")[0]
CU   = (torch.version.cuda or "cpu").replace(".", "")
TAG  = f"{PYV}_torch{TV}_cu{CU}"
ENV_TAR = f"{ENTORNOS}/pointcept_env_{TAG}.tar.gz"
VENV = "/content/pcenv"
PYBIN = f"{VENV}/bin/python"
PIP   = f"{VENV}/bin/pip"
POINTCEPT = "/content/Pointcept"
CAP = "%d.%d" % torch.cuda.get_device_capability(0)

if not os.path.isdir(POINTCEPT):
    sh(f"git clone --depth 1 https://github.com/Pointcept/Pointcept {POINTCEPT}")

if os.path.exists(ENV_TAR) and not os.path.exists(PYBIN):
    print(f"⚡ Restaurando entorno desde Drive: {ENV_TAR}")
    sh(f'tar -xzf "{ENV_TAR}" -C /content')
elif not os.path.exists(PYBIN):
    print(f"🔧 Entorno {TAG} no está en Drive. Construyéndolo (15-25 min la primera vez)...")
    sh("pip install -q virtualenv")
    # --system-site-packages: reutiliza el torch de Colab (no se reinstala)
    sh(f"virtualenv -q --system-site-packages {VENV}")
    sh(f"{PIP} install -q setuptools wheel ninja")
    sh(f"{PIP} install -q addict yapf einops h5py plyfile termcolor timm tensorboardX tensorboard "
       f"wandb SharedArray open3d ftfy regex tqdm huggingface_hub scipy pyyaml packaging")
    if CU.startswith("11"):
        spc = "spconv-cu118"
    elif CU.startswith("12"):
        spc = "spconv-cu120"
    else:
        spc = "spconv-cu126"   # no hay ruedas de spconv para CUDA 13 (Colab nuevo); es un intento
        print(f"⚠️ CUDA {torch.version.cuda} y Python {sys.version.split()[0]} son más nuevos que lo que soporta spconv. "
              "Puede fallar; la comprobación de abajo dirá qué módulo.")
    sh(f"{PIP} install -q {spc}")
    # torch-scatter: rueda precompilada de PyG si existe; si no, se compila
    url = f"https://data.pyg.org/whl/torch-{TV}+cu{CU}.html"
    r = subprocess.run(f"{PIP} install -q torch-scatter --only-binary=:all: -f {url}", shell=True, executable="/bin/bash")
    if r.returncode != 0:
        print("No hay rueda de torch-scatter para esta versión de torch; compilando (~10 min)...")
        sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-scatter")
    # pointops (extensión CUDA de Pointcept)
    sh(f"cd {POINTCEPT}/libs/pointops && TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation .")
    # el resto de dependencias que importa pointcept.models / pointcept.datasets
    sh(f"{PIP} install -q peft transformers torch-geometric")
    # torch-cluster: no hay rueda para torch tan nuevo; se compila (~5-10 min)
    sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-cluster")
    _recien_construido = True

# pointgroup_ops (segmentacion de instancias) no se usa en el preentrenamiento de Sonata: modulo vacio
os.makedirs("/content/stubs", exist_ok=True)
open("/content/stubs/pointgroup_ops.py", "w").write("# stub: no se usa en Sonata")

# --- Comprobación módulo por módulo (así se ve CUÁL falla y por qué) ---
ENV = dict(os.environ, PYTHONPATH=f"{POINTCEPT}:/content/stubs")
MODULOS = ["torch", "spconv.pytorch", "torch_scatter", "pointops", "timm", "addict", "wandb",
           "tensorboardX", "open3d", "torch_cluster", "torch_geometric", "peft", "transformers", "pointcept.models", "pointcept.datasets"]
fallos = {}
for m in MODULOS:
    r = subprocess.run([PYBIN, "-c", f"import {m}"], cwd=POINTCEPT, env=ENV, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  ✅ {m}")
    else:
        ultima = (r.stderr.strip().splitlines() or ["(sin mensaje)"])[-1]
        fallos[m] = ultima
        print(f"  ❌ {m}: {ultima}")

if fallos:
    print("\nENTORNO INCOMPLETO. Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
    print("No se guardó en Drive. Si ya había una copia mala en entornos/, bórrala:", ENV_TAR)
    raise RuntimeError("Faltan módulos: " + ", ".join(fallos))

if globals().get("_recien_construido"):
    print("💾 Entorno OK. Guardando en Drive para la próxima vez...")
    sh(f'tar -czf "{ENV_TAR}" -C /content pcenv')
print("✅ Entorno listo.")

In [ ]:
# @title 3b. Comprobar que el entorno funciona en ESTA GPU (si no, recompila y actualiza la caché en Drive)
import subprocess, os

PRUEBA = (
    "import torch, torch_scatter, pointops, torch_cluster\n"
    "x = torch.randn(8, device='cuda'); b = torch.tensor([0,0,1,1,2,2,3,3], device='cuda')\n"
    "torch_scatter.segment_coo(x, b, reduce='min'); torch.cuda.synchronize(); print('ok')\n"
)
open("/content/prueba_gpu.py", "w").write(PRUEBA)
def prueba():
    r = subprocess.run([PYBIN, "/content/prueba_gpu.py"], capture_output=True, text=True, env=ENV, cwd=POINTCEPT)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

ok, err = prueba()
if ok:
    print(f"✅ Entorno OK en {torch.cuda.get_device_name(0)} (capacidad {CAP}).")
else:
    print("⚠️ Las extensiones compiladas no corren en esta GPU:", err)
    print("   Recompilando para la capacidad", CAP, "(5-10 min). Después queda guardado en Drive.")
    ARCH = f"{CAP}+PTX"
    for pkg in ("torch-scatter", "torch-cluster"):
        sh(f"TORCH_CUDA_ARCH_LIST='{ARCH}' FORCE_CUDA=1 {PIP} install -q --no-build-isolation --no-deps --force-reinstall --no-cache-dir {pkg}")
    sh(f"cd {POINTCEPT}/libs/pointops && rm -rf build dist *.egg-info && TORCH_CUDA_ARCH_LIST='{ARCH}' FORCE_CUDA=1 "
       f"{PIP} install -q --no-build-isolation --no-deps --force-reinstall --no-cache-dir .")
    ok, err = prueba()
    assert ok, f"Sigue fallando tras recompilar: {err}"
    sh(f'tar -czf "{ENV_TAR}" -C /content pcenv')
    print("💾 Entorno recompilado y guardado en Drive para esta GPU.")
sh(f"{PIP} install -q laspy")     # lo necesita preprocesar.py (M1) y preparar_etiquetas.py

In [ ]:
# @title 4. Datos etiquetados: descomprimir el zip, unir las piezas y guardar las etiquetas
import os, glob, shutil, zipfile

RAW = "/content/etiquetado_raw"
CACHE_ENT = f"{M1_ENT}/{ESCENA}"; os.makedirs(CACHE_ENT, exist_ok=True)
MERGED_LAS = f"{WORK}/{ESCENA}.las"
ETIQ_NPZ = f"{WORK}/{ESCENA}_etiquetas.npz"
c_las, c_npz = f"{CACHE_ENT}/{ESCENA}.las", f"{CACHE_ENT}/{ESCENA}_etiquetas.npz"

if os.path.exists(c_las) and os.path.exists(c_npz) and not REHACER:
    print("⚡ Escena ya unida en Drive; se copia.")
    shutil.copy2(c_las, MERGED_LAS); shutil.copy2(c_npz, ETIQ_NPZ)
else:
    if not os.path.exists(ZIP_ETIQUETADO):
        print("No encuentro el zip en Drive. Súbelo ahora (queda guardado para la próxima):")
        from google.colab import files
        sub = files.upload()
        os.makedirs(os.path.dirname(ZIP_ETIQUETADO), exist_ok=True)
        shutil.move(next(iter(sub)), ZIP_ETIQUETADO)
    shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
    with zipfile.ZipFile(ZIP_ETIQUETADO) as zf:
        for m in zf.namelist():
            destino = os.path.realpath(os.path.join(RAW, m))
            assert destino.startswith(os.path.realpath(RAW)), f"ruta sospechosa en el zip: {m}"
        zf.extractall(RAW)
    print("Archivos .las:", len(glob.glob(f"{RAW}/**/*.las", recursive=True)))
    !{PYBIN} {REPO_DIR}/experimentacion/evaluacion_etiquetada/preparar_etiquetas.py unir --carpeta {RAW} --salida-las {MERGED_LAS} --salida-etiq {ETIQ_NPZ}
    shutil.copy2(MERGED_LAS, c_las); shutil.copy2(ETIQ_NPZ, c_npz)

In [ ]:
# @title 5. M1 · Preprocesar la escena (SOR + voxel 5 cm + normales) y pasar las etiquetas al resultado
import os, shutil

NPY = f"{WORK}/{ESCENA}.npy"
CACHE_M1 = f"{M1_SAL}/{ESCENA}"; os.makedirs(CACHE_M1, exist_ok=True)
archivos = [f"{ESCENA}.npy", f"{ESCENA}.json", f"{ESCENA}_etiquetas.npy", f"{ESCENA}_etiquetas.json"]
if all(os.path.exists(f"{CACHE_M1}/{a}") for a in archivos) and not REHACER:
    print("⚡ M1 ya calculado en Drive; se copia.")
    for a in archivos: shutil.copy2(f"{CACHE_M1}/{a}", f"{WORK}/{a}")
else:
    !{PYBIN} {REPO_DIR}/preprocesar.py --input {MERGED_LAS} --output {WORK}/{ESCENA}
    !{PYBIN} {REPO_DIR}/experimentacion/evaluacion_etiquetada/preparar_etiquetas.py mapear --npy {NPY} --json {WORK}/{ESCENA}.json --etiquetas {ETIQ_NPZ}
    for a in archivos: shutil.copy2(f"{WORK}/{a}", f"{CACHE_M1}/{a}")
import numpy as np
S = np.load(NPY, mmap_mode="r"); print("M1:", S.shape)

## M2 · HyperNCD (entorno aparte: Python 3.8 + MinkowskiEngine)
La celda 6 es la misma que usas en `colab_runner.ipynb`: restaura `entornos/hyperncd_env_cache.tar.gz` de Drive (~20 s) o lo compila la primera vez.

In [ ]:
# @title 6. M2 · Instalar o Restaurar hyperncd-env + MinkowskiEngine (Guarda caché en Drive como imagen lista)
%%bash
set -e

ENV_CACHE="/content/drive/MyDrive/PFC1_HyperNCD_Sonata/entornos/hyperncd_env_cache.tar.gz"

if [ ! -d "/content/miniconda3" ]; then
    echo "[1/4] Instalando Miniconda3 base..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Si ya existe la imagen del entorno compilada en tu Google Drive, la restauramos en ~20 segundos
if [ -f "$ENV_CACHE" ] && [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "⚡ Encontrado entorno pre-compilado en tu Drive ($ENV_CACHE). Restaurando en ~20 segundos..."
    mkdir -p /content/miniconda3/envs/hyperncd-env
    tar -xzf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env
    /content/miniconda3/envs/hyperncd-env/bin/python -c "import torch, MinkowskiEngine as ME; print('✅ Entorno restaurado desde Drive en segundos! MinkowskiEngine:', ME.__version__)"
    exit 0
fi

if [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "[2/4] Creando entorno hyperncd-env (Python 3.8 + CUDA 11.8 + OpenBLAS)..."
    conda create -y -n hyperncd-env python=3.8 -c conda-forge
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev
    conda install -y -n hyperncd-env -c conda-forge openblas-devel ninja gxx_linux-64=11
fi

conda activate hyperncd-env

echo "[3/4] Instalando PyTorch 1.10.1+cu113 y dependencias de HyperNCD..."
pip install -q torch==1.10.1+cu113 torchvision==0.11.2+cu113 -f https://download.pytorch.org/whl/torch_stable.html
pip install -q pytorch-lightning==1.4.8 torchmetrics==0.7.2 scipy==1.7.3 pyyaml tqdm scikit-learn plotly matplotlib
pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html

if ! python -c "import MinkowskiEngine" 2>/dev/null; then
    echo "[4/4] Compilando e instalando MinkowskiEngine (solo esta primera vez)..."
    # Asegurar cabecera nvtx3/nvToolsExt.h que pide MinkowskiEngine
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvtx-dev 2>/dev/null || true
    if [ ! -f "$CONDA_PREFIX/include/nvtx3/nvToolsExt.h" ]; then
        git clone -q --depth 1 https://github.com/NVIDIA/NVTX.git /tmp/NVTX
        cp -r /tmp/NVTX/c/include/nvtx3 "$CONDA_PREFIX/include/"
        rm -rf /tmp/NVTX
    fi
    if [ ! -d "/content/MinkowskiEngine" ]; then
        git clone -q https://github.com/NVIDIA/MinkowskiEngine.git /content/MinkowskiEngine
    fi
    cd /content/MinkowskiEngine
    export CUDA_HOME="$CONDA_PREFIX"
    export PATH="$CONDA_PREFIX/bin:$PATH"
    export CPLUS_INCLUDE_PATH="$CONDA_PREFIX/include:${CPLUS_INCLUDE_PATH:-}"
    export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
    export MAX_JOBS=4
    export CC="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc"
    export CXX="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++"
    python setup.py install --blas_include_dirs=${CONDA_PREFIX}/include --blas=openblas
fi

python -c "import torch, MinkowskiEngine as ME; print('✅ MinkowskiEngine listo, versión:', ME.__version__, '| GPU:', torch.cuda.get_device_name(0))"

# Guardar imagen del entorno ya compilado en tu Google Drive para no volver a compilar nunca más
if [ -d "/content/drive/MyDrive" ] && [ ! -f "$ENV_CACHE" ]; then
    echo "💾 Guardando imagen del entorno compilado en tu Drive ($ENV_CACHE) para que la próxima vez cargue en 20 segundos..."
    mkdir -p "$(dirname "$ENV_CACHE")"
    tar -czf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env .
    echo "✅ Imagen del entorno guardada en Drive."
fi

In [ ]:
# @title 7. M2 · f_sem de la escena con HyperNCD
import glob, os, re, shutil

PY_M2 = "/content/miniconda3/envs/hyperncd-env/bin/python"
FSEM = f"{WORK}/{ESCENA}_fsem.npz"
C_FSEM = f"{M2_SAL}/{ESCENA}/{ESCENA}_fsem.npz"; os.makedirs(os.path.dirname(C_FSEM), exist_ok=True)

if CKPT_M2 == "AUTO":
    ck = glob.glob(f"{M2_ENT}/**/*.ckpt", recursive=True)
    def _ep(c):
        m = re.search(r"epoch=(\d+)", os.path.basename(c)); return int(m.group(1)) if m else -1
    run4 = [c for c in ck if "split0_run4" in c and "best" in os.path.basename(c).lower()]
    cand = run4 or [c for c in ck if "best" in os.path.basename(c).lower()] or ck
    assert cand, f"No hay .ckpt en {M2_ENT}. Sube el checkpoint o pon CKPT_M2."
    CKPT_M2 = max(cand, key=_ep)
print("Checkpoint de M2:", CKPT_M2)

if os.path.exists(C_FSEM) and not REHACER:
    print("⚡ M2 ya calculado para esta escena y checkpoint?  (borra el archivo o usa REHACER=True si cambiaste de checkpoint)")
    shutil.copy2(C_FSEM, FSEM)
else:
    r = subprocess.run([PY_M2, f"{REPO_DIR}/predecir_hyperncd.py", "--input", NPY, "--ckpt", CKPT_M2, "--output", FSEM],
                       cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-4000:])
    assert r.returncode == 0, "Falló M2 (mira el error de arriba)"
    for ext in (".npz", ".json", ".csv"):
        shutil.copy2(FSEM.replace(".npz", ext), C_FSEM.replace(".npz", ext))

In [ ]:
# @title 8. M3 · f_ssl con Sonata para cada variante de M3_VARIANTES (original, adaptado, ...)
import os, shutil, subprocess

# Parche de memoria: atención SDPA en el Sonata del repo (sin flash-attn la atención densa no cabe)
f = f"{REPO_DIR}/sonata/sonata/model.py"
src = open(f, encoding="utf-8").read()
viejo = """            attn = (q * self.scale) @ k.transpose(-2, -1)  # (N', H, K, K)
            if self.enable_rpe:
                attn = attn + self.rpe(self.get_rel_pos(point, order))
            if self.upcast_softmax:
                attn = attn.float()
            attn = self.softmax(attn)
            attn = self.attn_drop(attn).to(qkv.dtype)
            feat = (attn @ v).transpose(1, 2).reshape(-1, C)
"""
orig_ind = "".join(("    " + l) if l.strip() else l for l in viejo.splitlines(True))
nuevo = """            if (not self.enable_rpe) and (not self.upcast_softmax):
                feat = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v.to(q.dtype),
                    dropout_p=self.attn_drop.p if self.training else 0.0,
                    scale=self.scale,
                )
                feat = feat.transpose(1, 2).reshape(-1, C).to(qkv.dtype)
            else:
""" + orig_ind
if "scaled_dot_product_attention" in src:
    print("Parche de atención ya aplicado.")
else:
    assert viejo in src, "No encontré el bloque de atención de Sonata (cambió el repo?)"
    open(f, "w", encoding="utf-8").write(src.replace(viejo, nuevo))
    print("✅ Parche de atención (SDPA) aplicado.")

ENV2 = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/sonata")
FSSL = {}
for nombre, pesos in M3_VARIANTES.items():
    salida = f"{WORK}/{ESCENA}_feats_{nombre}.npz"
    cache = f"{DRIVE}/m3/salida/{ESCENA}/{ESCENA}_feats_{nombre}.npz"; os.makedirs(os.path.dirname(cache), exist_ok=True)
    if os.path.exists(cache) and not REHACER:
        print(f"⚡ M3 '{nombre}' ya está en Drive; se copia (REHACER=True si cambiaste los pesos).")
        shutil.copy2(cache, salida)
    else:
        print(f"\n=== M3 '{nombre}'  pesos: {pesos or 'Sonata original (Meta)'} ===", flush=True)
        cmd = [PYBIN, "extraer_features.py", "--input", NPY, "--output", salida, "--semilla", "0"] + (["--pesos", pesos] if pesos else [])
        r = subprocess.run(cmd, cwd=REPO_DIR, env=ENV2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-2500:])
        assert r.returncode == 0, f"Falló M3 '{nombre}' (mira el error de arriba)"
        shutil.copy2(salida, cache)
    FSSL[nombre] = salida
print("\nVariantes de M3 listas:", list(FSSL))

In [ ]:
# @title 9. M4 · fusión y agrupamiento con cada variante de M3 (mismo M2, mismas regiones)
import os, shutil, subprocess

M4_DIRS = {}
for nombre, fssl in FSSL.items():
    sal = f"{WORK}/m4_{nombre}"; M4_DIRS[nombre] = sal
    if os.path.exists(f"{sal}/grupos_PL13.npz") and not REHACER:
        print(f"⚡ M4 '{nombre}' ya calculado en esta sesión."); continue
    print(f"\n=== M4 con M3 '{nombre}' ===", flush=True)
    r = subprocess.run([PYBIN, f"{REPO_DIR}/experimentacion/modulo4_real/correr_modulo4_real.py",
                        "--npy", NPY, "--fsem", FSEM, "--fssl", fssl, "--salida", sal, "--k"] + [str(k) for k in K_LISTA],
                       cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-2000:])
    assert r.returncode == 0, f"Falló M4 '{nombre}' (mira el error de arriba)"

In [ ]:
# @title 10. Evaluación con etiquetas (mIoU, ARI, NMI) y todo guardado en Drive
import os, shutil, subprocess
from IPython.display import Image, Markdown, display

EVAL = f"{WORK}/evaluacion"
grupos = [f"{n}={d}/grupos_PL13.npz" for n, d in M4_DIRS.items()]
r = subprocess.run([PYBIN, f"{REPO_DIR}/experimentacion/evaluacion_etiquetada/evaluar_m4.py",
                    "--etiquetas", f"{WORK}/{ESCENA}_etiquetas.npy", "--info", f"{WORK}/{ESCENA}_etiquetas.json",
                    "--grupos", *grupos, "--salida", EVAL], cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
assert r.returncode == 0, r.stdout[-3000:]
display(Markdown(open(f"{EVAL}/evaluacion.md", encoding="utf-8").read()))
display(Image(f"{EVAL}/evaluacion_miou.png"))

shutil.copytree(EVAL, f"{OUT_DRIVE}/evaluacion", dirs_exist_ok=True)
for n, d in M4_DIRS.items():
    shutil.copytree(d, f"{OUT_DRIVE}/m4_{n}", dirs_exist_ok=True)
print("\n💾 Resultados en Drive:", OUT_DRIVE)
!ls -lh "{OUT_DRIVE}"

In [ ]:
# @title 11. (Opcional) CONTROL de la adaptación: Sonata original y adaptado en el MISMO entorno, sobre PL13
# Mide cuánto cambia el descriptor SOLO por los pesos (mismo entorno, atención y semilla). Si el coseno es cercano a 1,
# la adaptación casi no movió el descriptor; si es bajo, el cambio es real y no viene del entorno.
HACER_CONTROL = False   # @param {type:"boolean"}
if HACER_CONTROL:
    import os, subprocess, numpy as np
    PL13_NPY = f"{REPO_DIR}/experimentacion/datos/modulo3_PL13/PL13_SCN0001.npy"
    assert os.path.exists(PL13_NPY), "Falta el .npy de PL13 en el repo clonado"
    ADAPT_PL13 = f"{DRIVE}/m3/salida/PL13_SCN0001_adaptado_feats.npz"
    assert os.path.exists(ADAPT_PL13), f"Falta {ADAPT_PL13} (el f_ssl adaptado de PL13 que ya sacaste)"
    ctrl = f"{WORK}/PL13_control_original.npz"
    r = subprocess.run([PYBIN, "extraer_features.py", "--input", PL13_NPY, "--output", ctrl, "--semilla", "0"],
                       cwd=REPO_DIR, env=ENV2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0
    a, b = np.load(ctrl), np.load(ADAPT_PL13)
    rng = np.random.default_rng(0); sel = np.sort(rng.choice(len(a["inverse"]), 200_000, replace=False))
    fa = a["feat"][a["inverse"][sel]].astype(np.float32); fb = b["feat"][b["inverse"][sel]].astype(np.float32)
    ca, cb = fa - fa.mean(0), fb - fb.mean(0)
    cos = (ca * cb).sum(1) / (np.linalg.norm(ca, axis=1) * np.linalg.norm(cb, axis=1) + 1e-8)
    print(f"Coseno centrado control(original) vs adaptado, mismo entorno: media {cos.mean():.3f}  (p5 {np.percentile(cos,5):.3f})")
    print("Compáralo con 0.54 (adaptado vs original de Khipu): si es parecido, el cambio viene de la adaptación y no del entorno.")
    shutil.copy2(ctrl, f"{DRIVE}/m3/salida/PL13_control_original_colab.npz")
else:
    print("Control omitido (HACER_CONTROL = False).")